# Sistema Inteligente de Recomendação para Redução do Desperdício Alimentar

## Projeto Aplicado III - Entrega 4

**Tema:** Sistema de recomendação de receitas alinhado ao ODS 12  
**Base:** Food.com Recipes and Interactions  
**Tipo de entrega:** Prova de conceito funcional com avaliação experimental

Este notebook segue a estrutura do Template do Módulo 4 e inclui trechos de código usados no projeto, além dos resultados calculados em `results/`.

# Sumário

1. Introdução
2. Referencial Teórico
3. Metodologia
4. Resultados
5. Métricas de Avaliação
6. Comparação com Baselines
7. Gráficos e Visualizações
8. Exemplo de Recomendação Híbrida
9. Discussão dos Resultados
10. Conclusões e Trabalhos Futuros
11. Referências
12. Apêndices

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
GRAFICOS_DIR = RESULTS_DIR / "graficos"
SRC_DIR = PROJECT_ROOT / "src"

def carregar_csv(nome):
    return pd.read_csv(RESULTS_DIR / nome)

def mostrar_grafico(nome):
    display(Image(filename=str(GRAFICOS_DIR / nome)))

def mostrar_codigo(arquivo, inicio=None, fim=None):
    linhas = (SRC_DIR / arquivo).read_text(encoding="utf-8").splitlines()
    inicio = 1 if inicio is None else inicio
    fim = len(linhas) if fim is None else fim
    trecho = "\n".join(linhas[inicio - 1:fim])
    display(Markdown(f"```python\n{trecho}\n```"))

def mostrar_funcao(arquivo, nome):
    linhas = (SRC_DIR / arquivo).read_text(encoding="utf-8").splitlines()
    inicio = None
    for i, linha in enumerate(linhas):
        if linha.startswith(f"def {nome}") or linha.startswith(f"class {nome}"):
            inicio = i
            break
    if inicio is None:
        raise ValueError(f"{nome} nao encontrado em {arquivo}")
    fim = len(linhas)
    for j in range(inicio + 1, len(linhas)):
        if linhas[j].startswith("def ") or linhas[j].startswith("class "):
            fim = j
            break
    trecho = "\n".join(linhas[inicio:fim])
    display(Markdown(f"```python\n{trecho}\n```"))

# 1. Introdução

## Contexto do Trabalho

O desperdício alimentar é um problema ligado ao uso ineficiente de recursos e se relaciona ao **ODS 12 - Consumo e Produção Responsáveis**.

## Motivação

A ideia do projeto é ajudar o usuário a transformar ingredientes disponíveis em casa em sugestões de receitas.

## Objetivo Geral

Desenvolver uma prova de conceito de sistema de recomendação de receitas para apoiar o aproveitamento de ingredientes disponíveis e contribuir para a redução do desperdício alimentar.

## Código relacionado

O trecho abaixo mostra a configuração da entrada simulada de validade/urgência usada no projeto.

In [ ]:
mostrar_codigo("evaluation.py", 18, 23)

# 2. Referencial Teórico

Foram usadas três abordagens de sistemas de recomendação:

- **Filtragem colaborativa:** usa padrões de interação entre usuários e receitas.
- **Filtragem baseada em conteúdo:** usa características das receitas, neste caso os ingredientes.
- **Modelo híbrido:** combina sinais colaborativos, conteúdo e urgência simulada de validade.

O código abaixo mostra as bibliotecas centrais usadas para SVD e TF-IDF.

In [ ]:
mostrar_codigo("collaborative_svd.py", 1, 5)
mostrar_codigo("content_based.py", 1, 5)

# 3. Metodologia

## Coleta de Dados

A leitura dos dados foi feita diretamente do `archive.zip`, usando os arquivos `RAW_recipes.csv` e `RAW_interactions.csv`.

In [ ]:
mostrar_funcao("data_loader.py", "ler_csv_do_zip")

## Pré-processamento e Amostra Filtrada

Para viabilizar a execução local, foi criada uma amostra filtrada:

- remoção de ratings nulos;
- usuários com pelo menos 5 interações;
- receitas com pelo menos 10 avaliações;
- limite configurável de usuários e receitas;
- `random_state=42` para reprodutibilidade.

In [ ]:
mostrar_funcao("data_loader.py", "preparar_amostra_modelagem")

## Divisão Treino/Teste por Usuário

A avaliação top-k foi feita separando interações de cada usuário entre treino e teste.

In [ ]:
mostrar_funcao("data_loader.py", "dividir_treino_teste_por_usuario")

# 4. Resultados

## Resultados da EDA

Os números abaixo foram calculados pelo script `src/eda.py` e salvos em `results/resumo_eda.csv`.

In [ ]:
mostrar_funcao("eda.py", "calcular_eda")

In [ ]:
resumo_eda = carregar_csv("resumo_eda.csv")
resumo_eda

In [ ]:
distribuicao = carregar_csv("distribuicao_avaliacoes.csv")
distribuicao

## Amostra de Modelagem

A tabela abaixo documenta a amostra usada nos modelos de ranking.

In [ ]:
resumo_amostra = carregar_csv("resumo_amostra_modelagem.csv")
resumo_amostra

# 5. Métricas de Avaliação

## RMSE e MAE

Foram usados para avaliar os baselines de predição de rating.

In [ ]:
mostrar_funcao("evaluation.py", "calcular_rmse")
mostrar_funcao("evaluation.py", "calcular_mae")

## Precision@K, Recall@K, F1@K, NDCG@K e HitRate@K

Foram usados para avaliar recomendações top-10. No teste, itens com `rating >= 4` foram considerados relevantes.

In [ ]:
mostrar_funcao("evaluation.py", "metricas_topk")

# 6. Comparação com Baselines

## Baselines de Rating

Os baselines usam média global, média por usuário e média por receita.

In [ ]:
mostrar_funcao("baseline_models.py", "prever_ratings_baselines")

In [ ]:
metricas_baseline = carregar_csv("metricas_baseline.csv")
metricas_baseline

## Baseline de Popularidade

O baseline de popularidade recomenda receitas mais frequentes no treino, excluindo itens já vistos pelo usuário.

In [ ]:
mostrar_funcao("baseline_models.py", "recomendar_por_popularidade")

# 7. Modelos Implementados

## Filtragem Colaborativa com SVD

O modelo colaborativo cria uma matriz usuário-receita esparsa e aplica `TruncatedSVD`.

In [ ]:
mostrar_funcao("collaborative_svd.py", "CollaborativeSVDRecommender")

## Filtragem Baseada em Conteúdo com TF-IDF

O modelo transforma ingredientes em texto, aplica TF-IDF e cria um perfil médio do usuário a partir das receitas bem avaliadas (`rating >= 4`).

In [ ]:
mostrar_funcao("content_based.py", "ContentBasedRecommender")

## Modelo Híbrido

O modelo híbrido combina score colaborativo, score de conteúdo e score de validade simulada, usando pesos configuráveis.

In [ ]:
mostrar_funcao("hybrid_recommender.py", "calcular_score_validade")
mostrar_funcao("hybrid_recommender.py", "HybridRecommender")

# 8. Métricas de Ranking dos Modelos

A tabela abaixo compara Popularidade, Conteúdo TF-IDF, Colaborativo SVD e Híbrido com métricas top-10.

In [ ]:
mostrar_funcao("evaluation.py", "run_ranking_evaluation")

In [ ]:
metricas_ranking = carregar_csv("metricas_ranking_modelos.csv")
metricas_ranking

# 9. Gráficos e Visualizações

Os gráficos abaixo foram gerados automaticamente e salvos em `results/graficos/`.

In [ ]:
mostrar_funcao("evaluation.py", "salvar_graficos_ranking")

In [ ]:
mostrar_grafico("distribuicao_avaliacoes.png")
mostrar_grafico("comparacao_rmse_mae.png")
mostrar_grafico("comparacao_precision_recall_ndcg.png")
mostrar_grafico("comparacao_modelos_ranking.png")

# 10. Exemplo de Recomendação Híbrida

O exemplo abaixo mostra recomendações para um usuário da amostra, com score final e justificativa de ingredientes encontrados.

In [ ]:
mostrar_funcao("recommender.py", "run_hybrid_recommendation_example")

In [ ]:
recomendacoes = carregar_csv("recomendacoes_hibridas_exemplo.csv")
colunas = [
    "user_id",
    "recipe_id",
    "recipe_name",
    "score_final",
    "score_colaborativo_norm",
    "score_conteudo_norm",
    "score_validade_norm",
    "matched_ingredients",
    "justificativa",
]
recomendacoes[colunas]

# 11. Aplicação Streamlit

Além do notebook, o projeto possui uma aplicação Streamlit simples para demonstração. Ela permite informar `user_id`, ingredientes disponíveis e urgência simulada de validade.

In [ ]:
mostrar_codigo("app_streamlit.py", 1, 60)

# 12. Discussão dos Resultados

A base Food.com possui grande volume de dados, mas também alta esparsidade. Isso torna a recomendação personalizada desafiadora.

Na amostra avaliada, o baseline de popularidade teve melhor desempenho nas métricas de ranking. Esse resultado é plausível porque a base apresenta forte concentração de ratings positivos e algumas receitas populares são boas candidatas gerais.

O modelo híbrido não superou a popularidade nas métricas principais, mas é o mais alinhado ao objetivo do projeto, pois combina personalização, ingredientes e urgência simulada de validade.

# 13. Conclusões e Trabalhos Futuros

## Conclusões

O projeto implementou uma prova de conceito funcional de recomendação de receitas alinhada ao ODS 12. Foram entregues EDA, baselines, SVD, TF-IDF, modelo híbrido, avaliação top-k, gráficos e aplicação Streamlit.

## Trabalhos Futuros

- Coletar dados reais de despensa e validade.
- Melhorar a normalização dos ingredientes.
- Testar amostras maiores.
- Ajustar os pesos do modelo híbrido.
- Avaliar com usuários reais.
- Incluir diversidade e novidade nas recomendações.

# 14. Referências

RICCI, F.; ROKACH, L.; SHAPIRA, B. Introduction to Recommender Systems Handbook. In: RICCI, F.; ROKACH, L.; SHAPIRA, B.; KANTOR, P. (org.). Recommender Systems Handbook. Boston: Springer, 2011.

SCIKIT-LEARN. Machine Learning in Python. Disponível em: https://scikit-learn.org/.

FOOD.COM Recipes and Interactions Dataset. Base de receitas e interações utilizada na prova de conceito.

# 15. Apêndices

## Artefatos

- Código-fonte: `src/`.
- Notebooks: `notebooks/`.
- Resultados: `results/`.
- Documentação: `docs/`.

## Links para preencher no documento final

- Link do GitHub: inserir após publicação.
- Link do vídeo: inserir após gravação.
- Link do dataset: inserir a fonte da base Food.com.